Reads data from NPC simulations and saves trajectories for each NTR.

In [8]:
# import dependencies
import RMF
import pickle
import numpy as np

In [25]:
def _has_depth_with_site(root, i):
    """ returns true if node subtree thru first child is at least i
        levels, including the root node itself, and the lead is a site """
#  print root, i, len(root.get_children())
    if (i==1) and root.get_name()=="site":
        return True
    c = root.get_children()
    if len(c) == 0:
        return False
    return _has_depth_with_site(c[0], i-1)

def _add_nodes(node, tf, type_prefixes, depth=0):
    '''
    node - rmf node to scan
    tf - typed factory
    type_prefixes - list of full type prefixes (e.g. "Nup1" for "Nup1N")

    adds only nodes whose type name begins with any of the specified type prefixes
    '''
    children = node.get_children()
    ret = []
    #print "inspecting", node.get_name()
    if len(children)==0:
        return ret
    if _has_depth_with_site(node, 3) and tf.get_is(children[0]):
        child_type = tf.get(children[0]).get_type_name()
        if any([child_type.startswith(tp) for tp in type_prefixes]):
            ret.append(children)
    for c in children:
        ret += _add_nodes(c, tf,  type_prefixes, depth+1)
    return ret


def load_data(input_rmf_path, kap_radius, kap_amount, start_t, end_t, step_t, frames_per_file=104, one_frame_from_each=False):
    trajectories = np.zeros(shape=(kap_amount, 3, int(((end_t - start_t) / step_t) * frames_per_file)))

    for rmf_t in range(start_t, end_t, step_t):
        in_fh = RMF.open_rmf_file_read_only(f"{input_rmf_path}/{rmf_t}.movie.rmf")
        rff = RMF.ReferenceFrameFactory(in_fh)
        tf = RMF.TypedFactory(in_fh)
        # fg_types = [f"fg{x}" for x in range(32)]
        kap_types = [f"kap{kap_radius}"]

        # load data
        type2chains={}
        for i, kap_type in enumerate(kap_types):
            type2chains[kap_type] = _add_nodes(in_fh.get_root_node(), tf, [kap_type])
            
        # set frame
        for f_id, f in enumerate(in_fh.get_frames()):
            in_fh.set_current_frame(f)
    
            traj_i = int(f_id + ((rmf_t - start_t) / step_t) * frames_per_file)
            # read data
            for kap_i in range(kap_amount):
                coord = rff.get(type2chains["kap35"][0][kap_i]).get_translation()
                
                trajectories[kap_i, 0, traj_i] = coord[0] / 10
                trajectories[kap_i, 1, traj_i] = coord[1] / 10
                trajectories[kap_i, 2, traj_i] = coord[2] / 10
            if one_frame_from_each:
                break
    return trajectories

def load_and_save_data(in_path, out_path):
    x_coords, y_coords, z_coords = load_data(in_path, 0)
    with open(out_path, "wb") as f:
        pickle.dump([x_coords, y_coords, z_coords], f)
    

In [27]:
trajectories = load_data(
          input_rmf_path="/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_50uM_1ns/2/",
          kap_radius=35,
          kap_amount=250,
          start_t=40000,
          end_t=80000,
          step_t=100,
          frames_per_file=1,
          one_frame_from_each=True
          )
with open("spatial_markov_chain_data.pickle", "wb") as f:
    pickle.dump(trajectories, f)